In [0]:
df_hospital_a = spark.read.parquet("/mnt/bronze/hospital-a/patients")

In [0]:
df_hospital_a.createOrReplaceTempView("patients_hospital_a")

In [0]:
df_hospital_b = spark.read.parquet("/mnt/bronze/hospital-b/patients")

In [0]:
df_hospital_b.createOrReplaceTempView("patients_hospital_b")

In [0]:
%sql
select * from patients_hospital_b LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cdm_patients AS
SELECT CONCAT(src_patient_id,'-',datasource) AS patient_key,*
FROM (
  SELECT 
    PatientID AS src_patient_id,
    FirstName,
    LastName,
    MiddleName,
    SSN,
    PhoneNumber,
    Gender,
    DOB,
    Address,
    ModifiedDate,
    datasource
    FROM patients_hospital_a
    UNION ALL
    SELECT 
    ID AS src_patient_id,
    F_Name,
    L_Name,
    M_Name,
    SSN,
    PhoneNumber,
    Gender,
    DOB,
    Address,
    Updated_Date AS ModifiedDate,
    datasource
    FROM patients_hospital_b
)

In [0]:
%sql
SELECT * FROM cdm_patients LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW quality_checks AS
SELECT
  patient_key,
  src_patient_id,
  FirstName,
  LastName,
  MiddleName,
  SSN,
  PhoneNumber,
  Gender,
  DOB,
  Address,
  ModifiedDate AS src_modified_date,
  datasource,
  CASE 
    WHEN src_patient_id IS NULL OR dob IS NULL OR firstname IS NULL OR LOWER(firstname) = 'null' THEN true
    ELSE FALSE
  END AS is_quarantined
FROM cdm_patients


In [0]:
%sql
select * from quality_checks where is_quarantined = true limit 10

In [0]:
%sql 
CREATE SCHEMA IF NOT EXISTS silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.patients(
  patient_key STRING,
  src_patient_id STRING,
  firstname STRING,
  lastname STRING,
  middlename STRING,
  ssn STRING,
  phonenumber STRING,
  gender STRING,
  dob DATE,
  address STRING,
  src_modified_date TIMESTAMP,
  datasource STRING,
  is_quarantined BOOLEAN,
  inserted_date TIMESTAMP,
  modified_date TIMESTAMP,
  is_current BOOLEAN
)
USING DELTA

In [0]:
%sql
MERGE INTO silver.patients AS target
USING quality_checks AS source
ON target.patient_key = source.patient_key
AND target.is_current = true
WHEN MATCHED
AND(
  target.src_patient_id != source.src_patient_id
  OR target.firstname != source.firstname
  OR target.lastname != source.lastname
  OR target.middlename != source.middlename
  OR target.ssn != source.ssn
  OR target.phonenumber != source.phonenumber
  OR target.gender != source.gender
  OR target.dob != source.dob
  OR target.address != source.address
  OR target.src_modified_date != source.src_modified_date
  OR target.datasource != source.datasource
  OR target.is_quarantined != source.is_quarantined
)
THEN UPDATE SET
  target.is_current = false,
  target.modified_date = current_timestamp()
WHEN NOT MATCHED
THEN INSERT (
  patient_key,
  src_patient_id,
  firstname,
  lastname,
  middlename,
  ssn, 
  phonenumber,
  gender,
  dob,
  address,
  src_modified_date,
  datasource,
  is_quarantined,
  inserted_date,
  modified_date,
  is_current
) 
VALUES(
  source.patient_key,
  source.src_patient_id,
  source.firstname,
  source.lastname,
  source.middlename,
  source.ssn,
  source.phonenumber,
  source.gender,
  source.dob,
  source.address,
  source.src_modified_date,
  source.datasource,
  source.is_quarantined,
  current_timestamp(),
  current_timestamp(),
  true
)

In [0]:
%sql
select * from silver.patients limit(10)